# One additional optimizer experiment

This notebook runs **one bounded part** and downloads it immediately. Use a fresh Colab GPU session for each part; already downloaded parts never need to be rerun.

In [ ]:
# Change only this value before each run. There are seven independent parts.
PART = "scheduler"
# Choices:
# scheduler
# width_2048
# width_4096_lr_01
# width_4096_lr_02
# width_4096_lr_03
# width_4096_lr_04
# width_4096_lr_05

## Checkout, test, and preflight

Select **Runtime → Change runtime type → GPU**, set `PART` above, and run all cells.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LokeshJatangi/transformer_optimizer_benchmark.git"
BRANCH = "colab-results"
WORKDIR = Path("/content/transformer-optimizer-benchmark")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
source_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
os.environ["SOURCE_COMMIT"] = source_commit
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["python", "-m", "pytest", "-q"], check=True)
print("Checked out source commit:", source_commit)

In [ ]:
import torch
from additional_experiments import ADDITIONAL_PARTS, WIDTH_4096_PARTS

assert PART in ADDITIONAL_PARTS, (PART, ADDITIONAL_PARTS)
assert torch.cuda.is_available(), "Enable a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB")
if PART in WIDTH_4096_PARTS:
    print("Width-4096 learning rate in this shard:", WIDTH_4096_PARTS[PART])
print("Running part:", PART)

## Run and persist this part

Logs are rewritten after every candidate run. The output folder is unique to this part.

In [ ]:
subprocess.run(["python", "additional_experiments.py", "--part", PART], check=True)

In [ ]:
import json

part_dir = WORKDIR / "results_additional_parts" / PART
saved = json.loads((part_dir / "metrics.json").read_text())
assert saved["provenance"]["part"] == PART
required = ["metrics.json", "run_log.json", "run_log.csv", "PART_SUMMARY.md"]
if PART == "scheduler":
    required.append("retained_model.pt")
missing = [name for name in required if not (part_dir / name).exists()]
assert not missing, missing
print((part_dir / "PART_SUMMARY.md").read_text())
print(f"Validated {len(saved['runs'])} logged runs.")

## Download now

Extract the zip at the repository root. It contains `results_additional_parts/<part>/`. Then start a fresh Colab session and select the next part.

In [ ]:
from google.colab import files

archive = shutil.make_archive(
    f"/content/{PART}", "zip", root_dir=WORKDIR,
    base_dir=str(Path("results_additional_parts") / PART),
)
print("Created", archive)
files.download(archive)